# SE3082 - Lab 10: CUDA in Google Colab
**Student ID: IT24101516**  
**Parallel Computing | Semester 2, 2026**

The notebook covers all four exercises. Exercises 1(b) and 1(c) use timed copies of my Lab 9 programs.
Comments with **IT24101516** mark the code additions and completed tasks.

Choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
All the source code is included here, so no other files need to be uploaded.
The larger programs take a few seconds to generate their random input values.

**Run status:** The GPU cells have not been executed yet. Run all cells in Colab to generate the results and timing tables,
then save or download the notebook so the outputs are included.


## Before starting: check the GPU
This checks the GPU and CUDA compiler, then chooses the matching architecture.
On a T4 the compile option is **-arch=sm_75**, as in the lab sheet.
The small helpers keep the write, compile, and run steps together and check each program's verification line.


In [ ]:
# IT24101516 - Check the runtime and prepare the compile/run helpers.
import csv
import json
import re
import shutil
import statistics
import subprocess
from pathlib import Path
from IPython.display import Markdown, display

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Choose a GPU under Runtime > Change runtime type, then run all cells again.')
print(subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True).stdout)
gpu_info = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
    check=True, capture_output=True, text=True
).stdout.strip().splitlines()[0]
gpu_name, compute_capability = [part.strip() for part in gpu_info.split(',')]
CUDA_ARCH = 'sm_' + compute_capability.replace('.', '')
NVCC = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
if not Path(NVCC).is_file():
    raise RuntimeError('CUDA compiler not found. Reconnect to a Colab GPU runtime.')
print(subprocess.run([NVCC, '--version'], check=True, capture_output=True, text=True).stdout)
print(f'GPU: {gpu_name}; architecture: {CUDA_ARCH}')

# IT24101516 - Compile each source after its %%writefile cell has run.
def compile_cuda(source_name):
    executable = Path(source_name).stem
    command = [NVCC, f'-arch={CUDA_ARCH}', source_name, '-o', executable]
    print(' '.join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    result.check_returncode()
    return executable

# IT24101516 - Save the output and check the results.
def run_program(executable, *, label=None, full_output=True, expected_error=None,
                verify=True):
    result = subprocess.run([f'./{executable}'], capture_output=True, text=True)
    output = result.stdout + result.stderr
    log_name = (label or executable) + '.txt'
    Path(log_name).write_text(output, encoding='utf-8')
    lines = output.splitlines()
    if full_output or len(lines) <= 14:
        print(output, end='')
    else:
        print('\n'.join(lines[:5]))
        print(f'... full output saved in {log_name} ...')
        print('\n'.join(lines[-5:]))
    if expected_error is not None:
        accepted = (expected_error,) if isinstance(expected_error, int) else tuple(expected_error)
        match = re.search(r'code:(\d+),', output)
        if result.returncode == 0 or not match or int(match.group(1)) not in accepted:
            raise AssertionError(f'Expected CUDA error in {accepted}; got exit {result.returncode}.')
        print(f'Expected failure caught (code {match.group(1)}). The working source is unchanged.\n')
    else:
        result.check_returncode()
        if verify:
            match = re.search(r'Verification: (\d+) mismatches out of (\d+)', output)
            if not match or int(match.group(1)) != 0:
                raise AssertionError('The program did not report zero mismatches.')
    return output

# IT24101516 - Read measured seconds from the C++ program's output.
def measured_time(output):
    match = re.search(r'Time taken - ([0-9.]+)', output)
    if not match:
        raise ValueError('No kernel timing was printed.')
    return float(match.group(1))


## Exercise 1(a) - Time the 512-block addition
Each block adds one pair of integers. The timer starts just before the launch and stops after
**cudaDeviceSynchronize()**. The measured interval includes the launch and the CPU's wait for the GPU.
The copy back to the CPU is outside this interval. All 512 answers are checked.


In [ ]:
%%writefile ex1_timed.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h>     // IT24101516 - Needed by gettimeofday().

#define N 512

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// IT24101516 - Each block adds one pair of values.
__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int count) {
    for (int i = 0; i < count; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // IT24101516 - Time the launch and wait for the completed kernel.
    double dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize(); // IT24101516 - Wait before stopping the clock.
    double dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE; // IT24101516 - Fail if verification fails.
}


In [ ]:
# IT24101516 - Compile and run the timed 512-block program.
compile_cuda('ex1_timed.cu')
ex1_output = run_program('ex1_timed')


### Experiment - leave out the synchronization
This makes a separate experimental file with the wait removed, so the working version stays ready to use.
I expect the printed interval to be shorter in many runs because it no longer waits for the kernel to finish.
These very small times can vary, so one run is not enough to prove a speed difference.

The answers should still verify: the later device-to-host **cudaMemcpy** waits for the result before returning.
That wait happens after the timer has stopped, so the reported interval no longer measures the completed calculation.


In [ ]:
# IT24101516 - Remove only the timed wait in a separate copy.
source = Path('ex1_timed.cu').read_text()
wait_line = '    cudaDeviceSynchronize(); // IT24101516 - Wait before stopping the clock.'
assert source.count(wait_line) == 1
source = source.replace(wait_line, '    // IT24101516 - Experiment: omit the wait before the clock stops.')
Path('ex1_no_sync.cu').write_text(source)
compile_cuda('ex1_no_sync.cu')
ex1_no_sync_output = run_program('ex1_no_sync')
print(f'With synchronization:    {measured_time(ex1_output):.7f} s')
print(f'Without synchronization: {measured_time(ex1_no_sync_output):.7f} s')


## Exercise 1(b) - Add timers to my Lab 9 programs
These are timed copies of **Exercises 3, 4, 5, and 6 from my Lab 9 notebook**.
Each program keeps its original calculation and launch configuration.
In particular, Exercise 5 multiplies the vectors, just as it did in Lab 9.
Each compile cell runs its program three times and records the measured kernel intervals.
Long result lists are saved in full in text files; a short preview is shown below each run.

**Prediction:** I expect one block of 512 threads to make better use of warp lanes than 512 blocks of one thread.
However, the one-block version uses only one streaming multiprocessor, and both calculations are tiny.
Launch overhead and timing noise can affect which looks faster.
The 100-million-element program processes ten times as many elements as the 10-million-element program,
but its time may not scale by exactly ten because memory bandwidth, GPU utilization, and indexing also matter.


In [ ]:
# IT24101516 - Start a fresh timing table whenever this section is rerun.
timing_results = {}


### My Lab 9 Exercise 3 - 512 blocks x 1 thread

Only the timer and its headers have been added to this copy.


In [ ]:
%%writefile ex3_blocks_timed.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h> // IT24101516 - Header for gettimeofday().

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // Each block adds one element
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory and generate random numbers
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy data from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch 512 blocks with 1 thread each
    // IT24101516 - Start the timer before the kernel launch.
    double dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize(); // IT24101516 - Wait before stopping the timer.
    double dElaps = cpuSecond() - dStart; // IT24101516 - Elapsed seconds.
    printf("Time taken - %.7f\n", dElaps); // IT24101516 - Print the measured time.

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy results from GPU to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print all 512 results
    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Verify results
    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Free memory
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
# IT24101516 - Compile my timed Exercise 3 and measure three separate runs.
compile_cuda('ex3_blocks_timed.cu')
times = []
for run in range(1, 4):
    print(f'Lab 9 Exercise 3, run {run}')
    output = run_program('ex3_blocks_timed', label=f'ex3_blocks_timed_run{run}', full_output=False)
    times.append(measured_time(output))
timing_results['Exercise 3'] = {'configuration': '512 blocks x 1 thread', 'times': times}


### My Lab 9 Exercise 4 - 1 block x 512 threads

Only the timer and its headers have been added to this copy.


In [ ]:
%%writefile ex4_threads_timed.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h> // IT24101516 - Header for gettimeofday().

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy data from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch 1 block with 512 threads
    // IT24101516 - Start the timer before the kernel launch.
    double dStart = cpuSecond();
    addT<<<1, N>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize(); // IT24101516 - Wait before stopping the timer.
    double dElaps = cpuSecond() - dStart; // IT24101516 - Elapsed seconds.
    printf("Time taken - %.7f\n", dElaps); // IT24101516 - Print the measured time.

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result from GPU to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print results
    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Verify results
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r])
            errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Free memory
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
# IT24101516 - Compile my timed Exercise 4 and measure three separate runs.
compile_cuda('ex4_threads_timed.cu')
times = []
for run in range(1, 4):
    print(f'Lab 9 Exercise 4, run {run}')
    output = run_program('ex4_threads_timed', label=f'ex4_threads_timed_run{run}', full_output=False)
    times.append(measured_time(output))
timing_results['Exercise 4'] = {'configuration': '1 block x 512 threads', 'times': times}


### My Lab 9 Exercise 5 - 10,000,000 elements; 19,532 blocks x 512 threads

Only the timer and its headers have been added to this copy.


In [ ]:
%%writefile ex5_vecmul_timed.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h> // IT24101516 - Header for gettimeofday().

#define N 10000000
#define THREADS_PER_BLOCK 512

// GPU kernel
__global__ void vecMul(int *a, int *b, int *c, int n) {

    // IT24101516 - Calculate global thread index
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // IT24101516 - Check array boundaries
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

// Generate random numbers
void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy data from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // IT24101516 - Calculate number of blocks
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    // IT24101516 - Launch the GPU kernel
    // IT24101516 - Start the timer before the kernel launch.
    double dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize(); // IT24101516 - Wait before stopping the timer.
    double dElaps = cpuSecond() - dStart; // IT24101516 - Elapsed seconds.
    printf("Time taken - %.7f\n", dElaps); // IT24101516 - Print the measured time.

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // IT24101516 - Copy results from GPU to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // IT24101516 - Print the last 1000 results
    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);
    }

    // IT24101516 - Verify all results
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Free CPU memory
    free(a);
    free(b);
    free(c);

    // Free GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
# IT24101516 - Compile my timed Exercise 5 and measure three separate runs.
compile_cuda('ex5_vecmul_timed.cu')
times = []
for run in range(1, 4):
    print(f'Lab 9 Exercise 5, run {run}')
    output = run_program('ex5_vecmul_timed', label=f'ex5_vecmul_timed_run{run}', full_output=False)
    times.append(measured_time(output))
timing_results['Exercise 5'] = {'configuration': '10,000,000 elements; 19,532 blocks x 512 threads', 'times': times}


### My Lab 9 Exercise 6 - 100,000,000 elements; 313 x 625 blocks; 32 x 16 threads

Only the timer and its headers have been added to this copy.


In [ ]:
%%writefile ex6_matrix_timed.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h> // IT24101516 - Header for gettimeofday().

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c] (NOT matrix multiplication)
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {

    // IT24101516 - Row comes from the y dimension, col from the x dimension
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // IT24101516 - Only compute if BOTH row and col are inside the matrix
    if (row < rows && col < cols) {

        // IT24101516 - Convert (row, col) into a flat index: row * cols + col
        int idx = row * cols + col;

        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // Host copies (flat arrays, row-major)
    int *d_a, *d_b, *d_c;       // Device copies

    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int); // 400 MB per matrix

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate input values
    a = (int *)malloc(size);
    random_ints(a, count);

    b = (int *)malloc(size);
    random_ints(b, count);

    c = (int *)malloc(size);

    // Copy inputs from host to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);

    // IT24101516 - Blocks needed in x (columns) and y (rows), rounding UP
    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    // IT24101516 - Launch mulElem with <<<numBlocks, threadsPerBlock>>>
    // IT24101516 - Start the timer before the kernel launch.
    double dStart = cpuSecond();
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    cudaDeviceSynchronize(); // IT24101516 - Wait before stopping the timer.
    double dElaps = cpuSecond() - dStart; // IT24101516 - Elapsed seconds.
    printf("Time taken - %.7f\n", dElaps); // IT24101516 - Print the measured time.

    // Check for kernel launch errors
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // IT24101516 - Copy d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // IT24101516 - Print C[9999][9000] to C[9999][9999] (last 1000 elements)
    for (int col = COLS - 1000; col < COLS; col++) {

        int idx = (ROWS - 1) * COLS + col;

        printf("C[%d][%d] = %d x %d = %d\n",
               ROWS - 1, col, a[idx], b[idx], c[idx]);
    }

    // IT24101516 - Verify all elements on the CPU and print mismatches
    int errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %zu\n", errors, count);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
# IT24101516 - Compile my timed Exercise 6 and measure three separate runs.
compile_cuda('ex6_matrix_timed.cu')
times = []
for run in range(1, 4):
    print(f'Lab 9 Exercise 6, run {run}')
    output = run_program('ex6_matrix_timed', label=f'ex6_matrix_timed_run{run}', full_output=False)
    times.append(measured_time(output))
timing_results['Exercise 6'] = {'configuration': '100,000,000 elements; 313 x 625 blocks; 32 x 16 threads', 'times': times}


### Measured timing table and comparison
These cells build the table from the three actual runs. The mean is included to make comparison easier.
The timer follows the sheet's **gettimeofday()** approach; very short intervals have limited precision.


In [ ]:
# IT24101516 - Show and save the three measured timings for every Lab 9 program.
table = ['| Lab 9 program | Launch configuration | Run 1 (s) | Run 2 (s) | Run 3 (s) | Mean (s) |',
         '|---|---|---:|---:|---:|---:|']
with open('lab9_kernel_timings_IT24101516.csv', 'w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['program', 'configuration', 'run1_s', 'run2_s', 'run3_s', 'mean_s'])
    for name in ['Exercise 3', 'Exercise 4', 'Exercise 5', 'Exercise 6']:
        item = timing_results[name]
        values = item['times']
        mean = statistics.mean(values)
        table.append(f'| {name} | {item["configuration"]} | ' +
                     ' | '.join(f'{v:.7f}' for v in values + [mean]) + ' |')
        writer.writerow([name, item['configuration'], *values, mean])
display(Markdown('\n'.join(table)))

# IT24101516 - Compare the measured times.
means = {name: statistics.mean(item['times']) for name, item in timing_results.items()}
print(f'512 blocks of one thread: {means["Exercise 3"]:.7f} s on average.')
print(f'One block of 512 threads: {means["Exercise 4"]:.7f} s on average.')
if means['Exercise 5'] > 0:
    print(f'Exercise 6 / Exercise 5 time ratio: {means["Exercise 6"] / means["Exercise 5"]:.2f}x.')
print('The data size increases by 10x, but these simple products are sensitive to memory bandwidth and launch overhead.')


## Exercise 1(c) - Kernel time compared with the full copy sequence
This is another copy of my **Lab 9 Exercise 5**, increased to **100,000,000 elements**.
The inner timer measures the kernel launch and synchronization.
The outer timer starts before the first host-to-device copy and stops after the device-to-host copy.
Allocation, random input generation, printing, and CPU verification are outside both intervals.

I expect the whole sequence to take longer because it also transfers two inputs and one output.
With four-byte integers this means about **1.2 GB** of data crossing between host and device.
A simple multiplication does very little work per element, so transfers can cost more than the calculation.
GPU work is more worthwhile when each transferred value is used for more computation, or when data can stay on the GPU between operations.


In [ ]:
%%writefile ex5_vecmul_transfers.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h> // IT24101516 - Header for gettimeofday().

#define N 100000000 // IT24101516 - Part (c): use 100 million elements.
#define THREADS_PER_BLOCK 512

// GPU kernel
__global__ void vecMul(int *a, int *b, int *c, int n) {

    // IT24101516 - Calculate global thread index
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // IT24101516 - Check array boundaries
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

// Generate random numbers
void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy data from CPU to GPU
    // IT24101516 - Set up and print the launch before either timed interval.
    // IT24101516 - Calculate number of blocks
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    // IT24101516 - Start the full-sequence timer before the first copy.
    double sequenceStart = cpuSecond();
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);



    // IT24101516 - Launch the GPU kernel
    // IT24101516 - Start the timer before the kernel launch.
    double dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize(); // IT24101516 - Wait before stopping the timer.
    double dElaps = cpuSecond() - dStart; // IT24101516 - Elapsed seconds.
    // IT24101516 - Print timings after the final copy so printing is not timed.

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // IT24101516 - Copy results from GPU to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
    double sequenceElapsed = cpuSecond() - sequenceStart; // IT24101516 - Stop after copy-out.
    printf("Time taken - %.7f\n", dElaps); // IT24101516 - Kernel interval.
    printf("Whole sequence - %.7f\n", sequenceElapsed); // IT24101516 - Copies plus kernel.

    // IT24101516 - Print the last 1000 results
    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);
    }

    // IT24101516 - Verify all results
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Free CPU memory
    free(a);
    free(b);
    free(c);

    // Free GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
# IT24101516 - Measure both intervals in the same execution and save the comparison.
compile_cuda('ex5_vecmul_transfers.cu')
transfer_output = run_program('ex5_vecmul_transfers', full_output=False)
kernel_seconds = measured_time(transfer_output)
sequence_seconds = float(re.search(r'Whole sequence - ([0-9.]+)', transfer_output).group(1))
copy_overhead = sequence_seconds - kernel_seconds
print(f'Kernel only: {kernel_seconds:.7f} s')
print(f'Whole sequence: {sequence_seconds:.7f} s')
print(f'Extra time for the copies and surrounding host work: {copy_overhead:.7f} s')
if kernel_seconds > 0:
    print(f'Whole sequence / kernel-only ratio: {sequence_seconds / kernel_seconds:.2f}x')
with open('kernel_vs_transfers_IT24101516.csv', 'w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['kernel_s', 'whole_sequence_s', 'extra_s'])
    writer.writerow([kernel_seconds, sequence_seconds, copy_overhead])


## Exercise 2 - Catch errors with CHECK()
All five starter tasks are completed below: the three allocations, both input copies, the launch check,
the synchronization, the output copy, and the three frees are checked.
**cudaGetLastError()** checks whether the launch was accepted; checking the synchronization also catches failures
that happen while the kernel runs.


In [ ]:
%%writefile ex2_check.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h>     // IT24101516 - Needed by gettimeofday().

#define N 512

// IT24101516 - Stop at a failed CUDA call and show where it happened.
#define CHECK(call)                                                     \
    do {                                                                \
        const cudaError_t error = (call);                                \
        if (error != cudaSuccess) {                                      \
            printf("Error: %s:%d, code:%d, reason: %s\n",                  \
                   __FILE__, __LINE__, (int)error,                       \
                   cudaGetErrorString(error));                          \
            exit(EXIT_FAILURE);                                         \
        }                                                               \
    } while (0)

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// IT24101516 - Each block adds one pair of values.
__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int count) {
    for (int i = 0; i < count; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    CHECK(cudaMalloc((void **)&d_a, size)); // IT24101516 - Check this runtime call.
    CHECK(cudaMalloc((void **)&d_b, size)); // IT24101516 - Check this runtime call.
    CHECK(cudaMalloc((void **)&d_c, size)); // IT24101516 - Check this runtime call.
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice)); // IT24101516 - Check this runtime call.
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice)); // IT24101516 - Check this runtime call.

    // IT24101516 - Time the launch and wait for the completed kernel.
    double dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    CHECK(cudaGetLastError()); // IT24101516 - Check the kernel launch.
    CHECK(cudaDeviceSynchronize()); // IT24101516 - Check completion before stopping the timer.
    double dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost)); // IT24101516 - Check this runtime call.
    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); // IT24101516 - Check the first device free.
    CHECK(cudaFree(d_b)); // IT24101516 - Check the second device free.
    CHECK(cudaFree(d_c)); // IT24101516 - Check the third device free.
    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE; // IT24101516 - Fail if verification fails.
}


In [ ]:
# IT24101516 - Confirm that the checked version still gives the right answers.
compile_cuda('ex2_check.cu')
ex2_output = run_program('ex2_check')


### Experiments A and B - make the errors happen on purpose
**A:** 2,048 threads in one block exceeds the limit of 1,024. The launch check should report
**code 9: invalid configuration argument**.
My saved Lab 9 run reported **invalid argument** for this launch, so this experiment also accepts
**code 1** if the current runtime reports it. The actual message is kept in the output.

**B:** Asking for **1 << 50 bytes** is far beyond the GPU's memory. The allocation check should report
**code 2: out of memory**. This request is rejected; it does not allocate that amount of memory.

The experiments run as separate programs, so their expected failures do not stop the notebook or change the working source.
A tests the kernel-launch row of the sheet's error table; B tests the runtime-call row.
Neither deliberately causes an error during execution, which is the third row checked by synchronization.


In [ ]:
# IT24101516 - Experiment A: change only the launch in a separate copy.
source = Path('ex2_check.cu').read_text()
launch = '    add<<<N, 1>>>(d_a, d_b, d_c);'
assert source.count(launch) == 1
invalid_source = source.replace(launch,
    '    add<<<N, 2048>>>(d_a, d_b, d_c); // IT24101516 - Deliberately invalid launch.')
Path('ex2_invalid_launch.cu').write_text(invalid_source)
compile_cuda('ex2_invalid_launch.cu')
run_program('ex2_invalid_launch', expected_error=(9, 1))

# IT24101516 - Experiment B: change only the first allocation in another copy.
allocation = 'CHECK(cudaMalloc((void **)&d_a, size));'
assert source.count(allocation) == 1
huge_source = source.replace(allocation,
    'CHECK(cudaMalloc((void **)&d_a, (size_t)1 << 50)); /* IT24101516 - Deliberately impossible allocation. */', 1)
Path('ex2_huge_allocation.cu').write_text(huge_source)
compile_cuda('ex2_huge_allocation.cu')
run_program('ex2_huge_allocation', expected_error=2)


### Experiment C - the same bad launch without CHECK()
This uses the plain Exercise 1 program. I clear the device output to zero before the experiment so that a failed launch
leaves a known value to inspect. The invalid launch cannot calculate the sums, but the unchecked program still reaches
its printing and verification code. Most results should therefore be wrong.
Without the verification loop or error checks, a printed time alone would not tell me the kernel had failed.

**Why do { ... } while (0)?** It makes **CHECK(call);** behave like one statement, including when it appears
inside an if without braces. The macro evaluates its argument once, then either continues or reports the error and exits.


In [ ]:
# IT24101516 - Run the bad launch without error checks in a separate file.
source = Path('ex1_timed.cu').read_text()
source = source.replace('    cudaMalloc((void **)&d_c, size);',
    '    cudaMalloc((void **)&d_c, size);\n'
    '    cudaMemset(d_c, 0, size); // IT24101516 - Known output for the failed-launch experiment.')
source = source.replace('    add<<<N, 1>>>(d_a, d_b, d_c);',
    '    add<<<N, 2048>>>(d_a, d_b, d_c); // IT24101516 - Invalid launch without CHECK().')
Path('ex2_no_check.cu').write_text(source)
compile_cuda('ex2_no_check.cu')
# IT24101516 - This launch should fail, so check the mismatch count below.
result = subprocess.run(['./ex2_no_check'], capture_output=True, text=True)
unchecked_output = result.stdout + result.stderr
print(unchecked_output, end='')
Path('ex2_no_check.txt').write_text(unchecked_output)
mismatch = re.search(r'Verification: (\d+) mismatches', unchecked_output)
assert mismatch and int(mismatch.group(1)) > 0, 'Expected incorrect results from the invalid launch.'
print('The verification loop exposes the failed calculation. CHECK() would have stopped at the launch.')

# IT24101516 - Confirm the original checked program still works after the experiments.
run_program('ex2_check')


## Exercise 3 - Scale an 800 x 600 picture with a 2D grid
The picture is one flat array of 480,000 **unsigned char** pixels.
Each thread doubles one pixel and caps the result at 255.
The CPU check applies the same rule.

**Prediction:** Values from 128 to 255 need the cap. That is 128 of the 256 possible input values,
so I expect about half the pixels, roughly 240,000, to be capped.
With 16 x 16 threads per block, the grid is **50 x 38** blocks.
It launches **486,400 threads**, of which **6,400** fall outside the image.
The last block row has eight valid rows and eight extra rows; the bounds check skips the extras.


In [ ]:
%%writefile ex3_picture.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h>     // IT24101516 - Needed by gettimeofday().

#define WIDTH 800
#define HEIGHT 600

// IT24101516 - Stop at a failed CUDA call and show where it happened.
#define CHECK(call)                                                     \
    do {                                                                \
        const cudaError_t error = (call);                                \
        if (error != cudaSuccess) {                                      \
            printf("Error: %s:%d, code:%d, reason: %s\n",                  \
                   __FILE__, __LINE__, (int)error,                       \
                   cudaGetErrorString(error));                          \
            exit(EXIT_FAILURE);                                         \
        }                                                               \
    } while (0)

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// IT24101516 - Double each pixel and cap the result at 255.
__global__ void PictureKernel(unsigned char *d_Pin, unsigned char *d_Pout,
                              int height, int width) {
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;
    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t size = count * sizeof(unsigned char);
    unsigned char *h_in = (unsigned char *)malloc(size);
    unsigned char *h_out = (unsigned char *)malloc(size);
    unsigned char *d_in, *d_out;
    // IT24101516 - Report a failed host allocation before using the arrays.
    if (!h_in || !h_out) {
        fprintf(stderr, "Host memory allocation failed.\n");
        free(h_in); free(h_out);
        return EXIT_FAILURE;
    }
    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // IT24101516 - Task 1: fill every pixel with a value from 0 to 255.
    for (size_t i = 0; i < count; i++)
        h_in[i] = (unsigned char)(rand() % 256);
    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    // IT24101516 - Task 2: use 16 threads in both block dimensions.
    dim3 threadsPerBlock(16, 16);
    // IT24101516 - Task 3: round both grid dimensions up.
    dim3 numBlocks((WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %u x %u blocks of %u x %u threads\n",
           numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);

    double dStart = cpuSecond();
    // IT24101516 - Task 4: launch with the two dim3 variables.
    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_in, d_out, HEIGHT, WIDTH);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    double dElaps = cpuSecond() - dStart;
    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    // IT24101516 - Task 5: show both complete top-left 50 x 50 portions.
    printf("Original picture: top-left 50 x 50\n");
    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++)
            printf("%3d ", (int)h_in[row * WIDTH + col]);
        printf("\n");
    }
    printf("Updated picture: top-left 50 x 50\n");
    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++)
            printf("%3d ", (int)h_out[row * WIDTH + col]);
        printf("\n");
    }

    // IT24101516 - Task 6: check all pixels and count those that needed the cap.
    size_t errors = 0, capped = 0;
    for (size_t i = 0; i < count; i++) {
        int expected = 2 * (int)h_in[i];
        if (expected > 255) {
            expected = 255;
            capped++;
        }
        if ((int)h_out[i] != expected) errors++;
    }
    printf("Verification: %zu mismatches out of %zu\n", errors, count);
    printf("Pixels capped at 255: %zu\n", capped);
    printf("Time taken - %.7f\n", dElaps);

    free(h_in); free(h_out);
    CHECK(cudaFree(d_in)); CHECK(cudaFree(d_out));
    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE; // IT24101516 - Return an error if any results do not match.
}


In [ ]:
# IT24101516 - Run the picture program and show both 50 x 50 sections.
compile_cuda('ex3_picture.cu')
picture_output = run_program('ex3_picture')
assert 'Grid: 50 x 38 blocks of 16 x 16 threads' in picture_output


## Exercise 4 - Multiply a 500 x 500 x 500 volume element by element
The flat arrays hold **125,000,000 integers each**, or about **500 MB per array**.
Together, the three host arrays use about 1.5 GB and the three device arrays use about 1.5 GB.
The program uses 8 x 8 x 8 = **512 threads per block** and **63 x 63 x 63 blocks**.

Each thread calculates its **x**, **y**, and **z** coordinates, checks all three bounds, and uses
**((size_t)z * ny + y) * nx + x** to reach its element.
This is a corresponding-element product. The last 1,000 values are printed in memory order,
starting at **C[499][498][0]** and ending at **C[499][499][499]**.


In [ ]:
%%writefile ex4_volume.cu
// IT24101516 - SE3082 Lab 10.
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h> // IT24101516 - CUDA runtime functions.
#include <sys/time.h>     // IT24101516 - Needed by gettimeofday().

#define NX 500
#define NY 500
#define NZ 500

// IT24101516 - Stop at a failed CUDA call and show where it happened.
#define CHECK(call)                                                     \
    do {                                                                \
        const cudaError_t error = (call);                                \
        if (error != cudaSuccess) {                                      \
            printf("Error: %s:%d, code:%d, reason: %s\n",                  \
                   __FILE__, __LINE__, (int)error,                       \
                   cudaGetErrorString(error));                          \
            exit(EXIT_FAILURE);                                         \
        }                                                               \
    } while (0)

// IT24101516 - Read the CPU wall clock in seconds for the lab timer.
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    // IT24101516 - Task 1: get the thread's position in all three dimensions.
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // IT24101516 - Task 2: skip every thread outside the volume.
    if (x < nx && y < ny && z < nz) {
        // IT24101516 - Task 3: x changes fastest, followed by y and then z.
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t count) {
    for (size_t i = 0; i < count; i++)
        x[i] = rand() % 100;
}

int main(void) {
    size_t count = (size_t)NX * NY * NZ;
    size_t size = count * sizeof(int);
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);
    // IT24101516 - Check host memory before filling these large arrays.
    if (!a || !b || !c) {
        fprintf(stderr, "Host memory allocation failed.\n");
        free(a); free(b); free(c);
        CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
        return EXIT_FAILURE;
    }
    random_ints(a, count);
    random_ints(b, count);
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(8, 8, 8);
    // IT24101516 - Task 4: round the number of blocks up in x, y, and z.
    dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
                   (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);
    printf("Grid: %u x %u x %u blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    double dStart = cpuSecond();
    // IT24101516 - Task 5: launch the element-wise 3D multiplication.
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    double dElaps = cpuSecond() - dStart;
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // IT24101516 - Task 6: print the last 1000 elements with their coordinates.
    for (size_t idx = count - 1000; idx < count; idx++) {
        int x = (int)(idx % NX);
        int y = (int)((idx / NX) % NY);
        int z = (int)(idx / ((size_t)NX * NY));
        printf("C[%d][%d][%d] = %d x %d = %d\n", z, y, x, a[idx], b[idx], c[idx]);
    }

    // IT24101516 - Task 7: verify every element against the CPU product.
    size_t errors = 0;
    for (size_t i = 0; i < count; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE; // IT24101516 - Return an error if any results do not match.
}


In [ ]:
# IT24101516 - Run the full volume and show all of its last 1000 results.
compile_cuda('ex4_volume.cu')
volume_output = run_program('ex4_volume')
assert 'Grid: 63 x 63 x 63 blocks' in volume_output
assert 'Verification: 0 mismatches out of 125000000' in volume_output


### Compare the 2D and 3D calculations
Both kernels read two integers, multiply them, and write one integer per element.
The 3D calculation has **1.25 times** as many elements as the 2D calculation, so the time ratio needs that context.
I would expect similar time per element if both keep the GPU's memory system busy.

The thread layout can still affect performance. With 32 x 16 blocks, a warp follows 32 adjacent columns.
With 8 x 8 x 8 blocks, a warp spans several shorter rows, which can change memory access efficiency.
Bounds checks and extra threads in the rounded-up grids also add work.
The comparison below uses the measured times rather than assuming that 3D is automatically faster or slower.


In [ ]:
# IT24101516 - Compare measured elapsed time and time per element.
time_2d = statistics.mean(timing_results['Exercise 6']['times'])
time_3d = measured_time(volume_output)
comparison = [
    '| Program | Elements | Kernel time (s) | Time per element (ns) |',
    '|---|---:|---:|---:|',
    f'| Lab 9 Exercise 6 (2D, mean of 3 runs) | 100,000,000 | {time_2d:.7f} | {time_2d / 100000000 * 1e9:.4f} |',
    f'| Lab 10 Exercise 4 (3D, 1 run) | 125,000,000 | {time_3d:.7f} | {time_3d / 125000000 * 1e9:.4f} |'
]
display(Markdown('\n'.join(comparison)))
if time_2d > 0:
    print(f'3D / 2D elapsed-time ratio: {time_3d / time_2d:.2f}x for 1.25x as many elements.')
print('The 3D value comes from one run, so small differences should not be treated as a firm ranking.')

# IT24101516 - Save the GPU details and timing results.
summary = {
    'student_id': 'IT24101516', 'gpu': gpu_name, 'architecture': CUDA_ARCH,
    'lab9_timings': timing_results,
    'kernel_vs_transfers': {'kernel_s': kernel_seconds, 'whole_sequence_s': sequence_seconds},
    'picture_kernel_s': measured_time(picture_output),
    'volume_kernel_s': time_3d
}
Path('lab10_results_IT24101516.json').write_text(json.dumps(summary, indent=2))
print('Measurements saved. Save/download this notebook now to keep the displayed outputs.')


## Finishing the lab
After **Run all** completes, check that the working programs report zero mismatches.
The intentional launch failure should show code 9 (or the runtime's code 1), the allocation failure should show code 2,
and the unchecked failure should show incorrect results.
The timing table should contain three measured values for each of the four Lab 9 programs.

Save the notebook, then use **File > Download > Download .ipynb** to get a copy with the outputs included.
When finished, choose **Runtime > Disconnect and delete runtime** to release the GPU.

The source programs are written to Colab's temporary disk when their cells run.
If that disk is cleared, running the notebook again recreates them.

**References:** the supplied Lab 10 sheet and my Lab 9 notebook.
Background on asynchronous launches and synchronization:
[NVIDIA CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/asynchronous-execution.html).
Notebook upload and download information:
[Google Colab FAQ](https://research.google.com/colaboratory/faq.html).
